In [ ]:
# Importacoes Necessarias

import requests

In [5]:
# Definimos a URL da API Olinda (Banco Central) apontando para o endpoint de Moedas
# Parâmetros: top=100 (limite), format=json, e select apenas das colunas de interesse

url = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/Moedas?$top=100&$format=json&$select=simbolo,nomeFormatado"

# Realizamos a requisição GET na API

reponse = requests.get(url)

# Boa Prática: Usamos .get('value', []) para extrair a lista de moedas com segurança.
# Se a API retornar um erro e a chave 'value' não existir, ele retorna uma lista vazia,
# evitando que o código quebre com um 'KeyError'

dados = reponse.json()['value']

# Verificamos se a API realmente retornou dados antes de acionar o Spark

if dados:
    # Criamos o DataFrame PySpark a partir do JSON recebido

    df = spark.createDataFrame(dados)

    # Renomeamos as colunas para o padrão do nosso modelo de dados (Masterdata)
    # usando o selectExpr para fazer a projeção e o 'AS' simultaneamente

    df = df.selectExpr(
        "nomeFormatado AS MoedaNome",
        "simbolo AS Moeda"
    )

    # Salvamos o DataFrame como uma tabela Delta (gerenciada) no Lakehouse
    # O modo 'overwrite' garante que a tabela seja recriada/atualizada a cada execução
    # refletindo sempre o estado mais atualizado da API do Banco Central

    df.write.mode('overwrite').saveAsTable('masterdata_moedas')

StatementMeta(, d40f270f-18eb-41fe-ab01-cd4d1374175e, 7, Finished, Available, Finished, False)